# 실습 3: SASRec 구현 (EDA + 어텐션 히트맵 + 라이브러리 구현)

이번 실습은 두 파트로 나뉩니다.
- **Part 1**: MovieLens 데이터를 **시간 순서**로 재구성하고, 시퀀스 길이·아이템 인기 분포를 탐색(EDA)합니다.
- **Part 2**: lab_01·02에서 배운 어텐션·트랜스포머 모듈을 조립해 **SASRec**을 만들고, 실제로 학습시켜 다음 영화를 추천합니다. 마지막에는 모델이 **어떤 과거 영화에 집중했는지**를 어텐션 히트맵으로 확인합니다.

**개념 복기 및 이론 점검**
- 4주차 협업 필터링은 평점을 '순서 없는 집합'으로 봤습니다. SASRec이 추가로 활용하는 신호는 무엇일까요?
- SASRec의 예측 점수는 '인코더 출력 × 아이템 임베딩'의 내적으로 계산됩니다. 왜 분류용 `Linear` 대신 아이템 임베딩을 재사용할까요?
- 학습 시 입력과 정답을 '한 칸 밀어서'(input=t까지, target=t+1) 만드는 이유는 무엇일까요?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Team-AnI/A-AND-I-4TH-AI-CODE-LAB/blob/main/5주차/lab_03_sasrec.ipynb)

> 위 배지를 누르면 이 노트북이 **여러분 Google 계정의 Colab**에서 열립니다. 수정본을 남기려면 `파일 → Drive에 사본 저장`.

## 1. 환경 설정 및 데이터 로딩
4주차와 동일한 MovieLens ml-100k를 사용합니다. 첫 실행 시 자동 다운로드됩니다.

In [ ]:
!wget -q https://files.grouplens.org/datasets/movielens/ml-100k.zip -O ml-100k.zip
!unzip -q -o ml-100k.zip

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else (
    "mps" if torch.backends.mps.is_available() else "cpu")

df = pd.read_csv('ml-100k/u.data', sep='\t', header=None,
                 names=['user_id', 'item_id', 'rating', 'timestamp'])
movies = pd.read_csv('ml-100k/u.item', sep='|', header=None, encoding='latin-1',
                     usecols=[0, 1], names=['item_id', 'title'])
movie_names = dict(zip(movies['item_id'], movies['title']))

n_items = int(df['item_id'].max())
print("평점 수:", len(df), "| 사용자 수:", df['user_id'].nunique(), "| 영화 수:", n_items)
print("device:", device)

---
# Part 1: EDA (시간 순서 데이터 탐색)

모델을 만들기 전에, **순차 추천에 쓸 데이터가 어떻게 생겼는지** 먼저 파악합니다.

## 2-1. 사용자 시퀀스 구성
협업 필터링은 평점을 집합으로 봤지만, SASRec은 **무엇 다음에 무엇을 봤는지** 순서가 핵심입니다.

In [ ]:
# 4주차에서는 '순서 없는 평점 집합'으로 협업 필터링을 했습니다.
# SASRec은 '시간 순서'가 핵심이므로, 각 사용자의 시청 이력을 timestamp로 정렬합니다.
df_sorted = df.sort_values(['user_id', 'timestamp'])
user_seqs = df_sorted.groupby('user_id')['item_id'].apply(list)

print("사용자 1번의 시청 순서 (앞 10개):", user_seqs[1][:10])
print("총 사용자 시퀀스 수:", len(user_seqs))

### 🔬 코드 해설
- **`sort_values(['user_id', 'timestamp'])`**: 각 사용자의 행동을 **시간 순**으로 정렬합니다. 이 순서가 곧 SASRec의 입력 시퀀스가 됩니다. 4주차에서는 무시했던 `timestamp` 컬럼이 여기서 핵심 신호로 살아납니다.
- **`groupby('user_id')['item_id'].apply(list)`**: 사용자별로 시청한 영화 ID를 시간 순 리스트로 묶습니다. 이 리스트 하나가 한 사용자의 '시청 이력 문장'에 해당합니다.

## 2-2. 시퀀스 길이와 아이템 인기 분포
시퀀스가 얼마나 긴지, 평점이 특정 영화에 얼마나 몰리는지 확인합니다.

In [ ]:
seq_lens = user_seqs.apply(len)
item_pop = df['item_id'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(seq_lens, bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(seq_lens.median(), color='red', linestyle='--',
                label=f'median {int(seq_lens.median())}')
axes[0].set_title('User Sequence Length Distribution')
axes[0].set_xlabel('Sequence Length (# movies watched)')
axes[0].set_ylabel('Number of Users')
axes[0].legend()

axes[1].plot(range(1, len(item_pop) + 1), item_pop.values, color='coral')
axes[1].set_title('Item Popularity (Long Tail)')
axes[1].set_xlabel('Movie Rank (인기순)')
axes[1].set_ylabel('Number of Ratings')

plt.tight_layout(); plt.show()

print(f"시퀀스 길이 — 최소 {seq_lens.min()}, 중앙값 {int(seq_lens.median())}, 최대 {seq_lens.max()}")

### 🔬 코드 해설
- **시퀀스 길이 분포**: 사용자마다 시청 이력 길이가 다릅니다. 너무 긴 시퀀스는 메모리·연산 부담이 되므로, Part 2에서 **최근 `maxlen`개**만 잘라 쓰는 근거가 됩니다.
- **롱테일(Long Tail)**: 4주차에서 본 것과 같은 현상입니다. 소수 인기 영화에 평점이 몰리고 대다수 영화는 평점이 적습니다. 순차 추천 모델도 이 편향에 영향을 받으므로, 추천 결과가 인기작에 쏠리지 않는지 늘 점검해야 합니다.

---
# Part 2: SASRec 구현 및 학습

lab_01·02에서 다룬 임베딩·셀프 어텐션·트랜스포머 인코더·causal mask를 **하나의 모델 클래스**로 조립합니다.

## 3-1. 학습 데이터 준비 (다음-아이템 예측)
각 시점에서 '바로 다음에 본 영화'를 정답으로 삼아, 시퀀스를 한 칸 밀어 입력/정답 쌍을 만듭니다.

In [ ]:
maxlen = 50   # 최근 50개 행동만 사용 (그보다 길면 잘라냄)

def make_io(seq, maxlen):
    seq = seq[-(maxlen + 1):]          # 최근 maxlen+1개만 사용
    inp, tgt = seq[:-1], seq[1:]       # 입력: t까지, 정답: t+1
    pad = maxlen - len(inp)
    return [0] * pad + inp, [0] * pad + tgt   # 왼쪽을 0(pad)으로 채움

inputs, targets = [], []
for seq in user_seqs:
    if len(seq) < 2:
        continue
    i, t = make_io(seq, maxlen)
    inputs.append(i); targets.append(t)

inputs = torch.tensor(inputs)     # (N, maxlen)
targets = torch.tensor(targets)   # (N, maxlen)
print("학습 입력 shape:", inputs.shape, "| 정답 shape:", targets.shape)
print("예시 입력(뒤 8개):", inputs[0][-8:].tolist())
print("예시 정답(뒤 8개):", targets[0][-8:].tolist())

### 🔬 코드 해설
- **입력/정답을 한 칸 밀기**: 입력이 `[A, B, C]`라면 정답은 `[B, C, D]`입니다. 즉 'A까지 봤을 때 다음은 B', 'A,B까지 봤을 때 다음은 C'를 **모든 시점에서 동시에** 학습합니다. causal mask가 미래를 가려주므로 한 번의 forward로 모든 시점을 안전하게 훈련할 수 있습니다.
- **왼쪽 패딩(0)**: 시퀀스 길이가 제각각이므로 `maxlen`에 맞춰 앞쪽을 0으로 채웁니다. 0은 '아이템 없음(pad)'을 뜻하며, 모델의 `padding_idx`와 손실의 `ignore_index`로 무시됩니다.
- **`maxlen`으로 자르기**: 최근 행동이 다음 행동을 더 잘 설명하므로, 오래된 이력은 잘라내고 최근 `maxlen`개만 사용합니다.

## 3-2. SASRec 모델 정의
아이템 임베딩 + 위치 임베딩 → causal 셀프 어텐션 인코더 → 아이템 임베딩과의 내적으로 점수 산출.

In [ ]:
class SASRec(nn.Module):
    def __init__(self, num_items, maxlen, d_model=64, nhead=2, num_layers=2, dropout=0.2):
        super().__init__()
        self.item_emb = nn.Embedding(num_items + 1, d_model, padding_idx=0)  # 0 = pad
        self.pos_emb = nn.Embedding(maxlen, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model, nhead, dim_feedforward=d_model * 2,
            dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers)
        self.maxlen = maxlen

    def forward(self, seq):                       # seq: (B, L)
        L = seq.size(1)
        positions = torch.arange(L, device=seq.device).unsqueeze(0)
        x = self.item_emb(seq) + self.pos_emb(positions)
        causal = torch.triu(torch.full((L, L), float('-inf'), device=seq.device), diagonal=1)
        h = self.encoder(x, mask=causal)          # (B, L, d_model)
        logits = h @ self.item_emb.weight.T       # 아이템 임베딩과 내적 → 점수 (B, L, num_items+1)
        return logits

model = SASRec(n_items, maxlen).to(device)
print(model)
print("학습 파라미터 수:", sum(p.numel() for p in model.parameters()))

### 🔬 코드 해설
- **`item_emb` / `pos_emb`**: lab_01·02와 동일한 구조입니다. `padding_idx=0`은 pad 토큰의 임베딩을 0으로 고정해 학습에서 제외합니다.
- **`TransformerEncoder` + `causal` 마스크**: lab_01·02에서 본 인과적 셀프 어텐션을 그대로 사용합니다. 각 시점은 자기 자신과 과거만 봅니다.
- **`h @ self.item_emb.weight.T`**: 예측을 위해 별도의 분류 레이어를 두지 않고 **아이템 임베딩을 재사용**합니다. 인코더 출력 벡터와 각 아이템 임베딩의 내적이 곧 그 아이템의 점수입니다. 이는 8주차에서 배운 '벡터 내적이 유사도'라는 아이디어의 연장이며, SASRec 논문이 채택한 방식입니다.
- 결과 `logits` shape `(B, L, num_items+1)`: 각 시점마다 모든 아이템에 대한 점수입니다.

## 3-3. 모델 학습
다음-아이템을 맞히는 분류 문제로 학습합니다. CPU로도 1~2분이면 끝납니다.

In [ ]:
ds = TensorDataset(inputs, targets)
dl = DataLoader(ds, batch_size=128, shuffle=True)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
vocab = n_items + 1

model.train()
for epoch in range(1, 6):
    total = 0.0
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        logits = model(xb)
        # pad(0) 위치는 손실에서 제외 (ignore_index=0)
        loss = F.cross_entropy(logits.reshape(-1, vocab), yb.reshape(-1), ignore_index=0)
        loss.backward(); opt.step()
        total += loss.item()
    print(f"epoch {epoch}  loss {total / len(dl):.4f}")

### 🔬 코드 해설
- **`cross_entropy(..., ignore_index=0)`**: 다음 아이템 예측을 (전체 아이템 중 하나를 고르는) 분류로 풉니다. `ignore_index=0`으로 pad 위치는 손실 계산에서 빠집니다.
- **`logits.reshape(-1, vocab)`**: `(B, L, vocab)`을 `(B*L, vocab)`으로 펴서 모든 시점을 한꺼번에 손실에 넣습니다. 정답도 같은 방식으로 폅니다.
- epoch가 진행되며 loss가 감소하면, 모델이 '이 영화들을 본 사람은 다음에 저 영화를 본다'는 패턴을 학습하고 있다는 신호입니다.

## 3-4. 다음 영화 추천
학습된 모델로 특정 사용자의 마지막 시점 출력을 보고 다음 영화를 추천합니다.

In [ ]:
model.eval()

def recommend(user_id, top_n=10):
    seq = user_seqs[user_id][-maxlen:]
    inp = torch.tensor([[0] * (maxlen - len(seq)) + seq], device=device)
    with torch.no_grad():
        logits = model(inp)[0, -1]          # 마지막 시점의 다음-아이템 점수
    logits[0] = -float('inf')               # pad 토큰 제외
    for i in seq:                           # 이미 본 영화 제외
        logits[i] = -float('inf')
    return torch.topk(logits, top_n).indices.tolist()

uid = 1
print(f"사용자 {uid}번이 최근 본 영화:")
for i in user_seqs[uid][-5:]:
    print("   -", movie_names.get(i, 'Unknown'))

print(f"\n사용자 {uid}번에게 추천하는 다음 영화 Top 10:")
for rank, item in enumerate(recommend(uid), 1):
    print(f"  {rank:2d}. {movie_names.get(item, 'Unknown')}")

### 🔬 코드 해설
- **`model(inp)[0, -1]`**: 시퀀스의 **마지막 시점** 출력만 사용합니다. 이것이 '이 사용자가 지금까지 본 순서를 고려할 때 다음에 볼 영화'에 대한 점수입니다.
- **이미 본 영화 제외**: 추천에서 이미 시청한 영화와 pad(0)는 점수를 `-inf`로 막아 제외합니다.
- 4주차 협업 필터링은 '비슷한 사람/아이템'으로 추천했지만, SASRec은 '**내 시청 순서의 흐름**'으로 다음을 예측합니다. 같은 사용자라도 본 순서가 달라지면 추천이 달라질 수 있습니다.

> 💡 **직접 해보기**: `uid`를 다른 사용자로 바꿔보세요. 최근 본 영화의 장르 흐름과 추천 결과가 어떤 관계인지 살펴보세요.

## 3-5. 어텐션 가중치 히트맵
모델이 다음 영화를 예측할 때 **과거의 어떤 영화에 집중**했는지 시각화합니다.

In [ ]:
# 학습된 첫 번째 인코더 층의 셀프 어텐션 가중치를 직접 추출합니다.
uid = 1
recent = user_seqs[uid][-20:]                     # 최근 20개만 (보기 좋게)
L = len(recent)
seq_t = torch.tensor([recent], device=device)

positions = torch.arange(L, device=device).unsqueeze(0)
x = model.item_emb(seq_t) + model.pos_emb(positions)
causal = torch.triu(torch.full((L, L), float('-inf'), device=device), diagonal=1)

layer0 = model.encoder.layers[0]
with torch.no_grad():
    _, attn_w = layer0.self_attn(x, x, x, attn_mask=causal,
                                 need_weights=True, average_attn_weights=True)

plt.figure(figsize=(7, 6))
plt.imshow(attn_w[0].cpu(), cmap='YlOrRd')
plt.colorbar(fraction=0.046)
plt.title(f'SASRec Attention Weights (user {uid}, layer 1)')
plt.xlabel('Key position (past movies)')
plt.ylabel('Query position (current step)')
plt.tight_layout(); plt.show()

### 🔬 코드 해설
- 학습된 첫 번째 인코더 층의 셀프 어텐션 가중치를 직접 뽑아 그렸습니다. 세로축(Query)은 각 예측 시점, 가로축(Key)은 참고된 과거 영화입니다.
- causal mask 때문에 **대각선 위쪽(미래)은 항상 0**입니다. 대각선 아래에서 특정 칸이 진하다면, 모델이 그 과거 영화를 다음 예측에 특히 중요하게 봤다는 뜻입니다.
- lab_01에서는 학습 전이라 가중치가 무의미했지만, 지금은 **학습된** 가중치이므로 '모델이 실제로 무엇을 보고 있는가'를 해석할 수 있습니다. 이것이 SASRec 논문이 강조한 어텐션의 **해석 가능성**입니다.

## 4. ✅ 학습 결과 정리

**Part 1 EDA에서:**
- 평점을 `timestamp`로 정렬해 사용자별 **시청 순서 시퀀스**를 구성했습니다.
- 시퀀스 길이 분포로 `maxlen`을 정하는 근거를, 롱테일 분포로 인기 편향을 확인했습니다.

**Part 2 SASRec에서:**
- lab_01·02의 임베딩·셀프 어텐션·causal mask·트랜스포머 인코더를 하나의 모델로 조립했습니다.
- 시퀀스를 한 칸 밀어 다음-아이템 예측으로 학습하고, 실제 추천을 생성했습니다.
- 학습된 어텐션 가중치 히트맵으로 모델이 과거의 어떤 영화에 집중하는지 해석했습니다.

🎯 **핵심 결론:** 8주차 임베딩 → 9주차 협업 필터링 → 10주차 셀프 어텐션 기반 순차 추천으로 이어지며, 추천 시스템은 '유사도 계산'에서 '행동 순서를 학습하는 딥러닝'으로 진화했습니다. 다음 단계인 Airflow(6주차)에서는 이런 모델 학습·추천 과정을 **자동화된 파이프라인**으로 엮는 법을 배웁니다. 오늘 만든 학습→추천 흐름이 거기서 하나의 DAG가 될 수 있음을 떠올려 보세요.